In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import random
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
# Random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)
# Informative sensors
informative_sensors = [
    'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
    'sensor_9', 'sensor_11', 'sensor_12', 'sensor_14',
    'sensor_15', 'sensor_20', 'sensor_21'
]
# Column names
column_names = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
# Load data, FD002 for pretraing and FD001 for fine tuning
Data_raw=Path('..')/'data'/'raw'
fd002 = pd.read_csv(Data_raw / 'train_FD002.txt', sep='\s+', header=None, names=column_names)
fd002['RUL'] = fd002.groupby('engine_id')['cycle'].transform(max) - fd002['cycle']
fd002['RUL'] = fd002['RUL'].clip(upper=125)

fd001 = pd.read_csv(Data_raw / 'train_FD001.txt', sep='\s+', header=None, names=column_names)
fd001['RUL'] = fd001.groupby('engine_id')['cycle'].transform(max) - fd001['cycle']
fd001['RUL'] = fd001['RUL'].clip(upper=125)

 # Split FD001 by engine_id 
engines= fd001['engine_id'].unique()
split_idx= int(len(engines)*0.8)

train_engines= engines[:split_idx]
val_engines= engines[split_idx:]
train_data= fd001[fd001['engine_id'].isin(train_engines)]
val_data= fd001[fd001['engine_id'].isin(val_engines)]
# Normalization
scaler = StandardScaler()
fd002_scaled = fd002.copy()
fd002_scaled[informative_sensors] = scaler.fit_transform(fd002[informative_sensors])
train_scaled = train_data.copy()
train_scaled[informative_sensors] = scaler.transform(train_data[informative_sensors])
val_scaled = val_data.copy()
val_scaled[informative_sensors] = scaler.transform(val_data[informative_sensors])
# Sequence creation for LSTM
def create_sequences(df, sequence_length=30):
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        for i in range(len(engine_data) - sequence_length):
            X.append(engine_data[informative_sensors].iloc[i:i+sequence_length].values)
            y.append(engine_data['RUL'].iloc[i+sequence_length])
    return np.array(X), np.array(y)
# Fd002 pretraining
X_fd002, y_fd002 = create_sequences(fd002_scaled)
# Limited fd001 for fine-tuning
def limited_data(df, fraction=0.3):
    limited = []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        n_cycles = int(len(engine_data) * fraction)
        limited.append(engine_data.head(n_cycles))
    return pd.concat(limited)
train_limited = limited_data(train_scaled, fraction=0.3)
X_train_limited, y_train_limited = create_sequences(train_limited)
X_val, Y_val = create_sequences(val_scaled)
# Pretraining on FD002
model = Sequential([
    LSTM(64, input_shape=(30,11)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_fd002, y_fd002, validation_data=(X_val, Y_val), epochs=20, batch_size=64, callbacks=[early_stopping])
# Fine-tuning on limited FD001
model.layers[0].trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='mse')
model.fit(X_train_limited, y_train_limited,
          validation_data=(X_val, Y_val),
          epochs=20, batch_size=64,
          callbacks=[early_stopping])
y_pred=model.predict(X_val).flatten()
rmse= root_mean_squared_error(Y_val, y_pred)
r2= r2_score(Y_val, y_pred)
print(f'Trasfer Learning RMSE: {rmse:.2f}, R2:{r2:.2f}')
# Test with different limited data fractions
for fraction in [0.2, 0.3, 0.5, 0.7]:
    train_limited = limited_data(train_scaled, fraction=fraction)
    X_train_limited, y_train_limited = create_sequences(train_limited)
    
    # New model for each fraction to ensure fair comparison
    model_test = Sequential([
        LSTM(64, input_shape=(30, 11)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model_test.compile(optimizer='adam', loss='mse')
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    model_test.fit(X_fd002, y_fd002,
                   validation_data=(X_val, Y_val),
                   epochs=20, batch_size=64,
                   callbacks=[early_stop], verbose=0)
    
    model_test.layers[0].trainable = False
    model_test.compile(optimizer='adam', loss='mse')
    
    model_test.fit(X_train_limited, y_train_limited,
                   validation_data=(X_val, Y_val),
                   epochs=20, batch_size=64,
                   callbacks=[early_stop], verbose=0)
    
    y_pred = model_test.predict(X_val, verbose=0).flatten()
    rmse = root_mean_squared_error(Y_val, y_pred)
    r2 = r2_score(Y_val, y_pred)
    print(f'Fraction {fraction}: RMSE={rmse:.2f}, R2={r2:.2f}')


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - loss: 2415.6755 - val_loss: 1766.1299
Epoch 2/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - loss: 1694.2883 - val_loss: 4964.4595
Epoch 3/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - loss: 573.9751 - val_loss: 1497.2227
Epoch 4/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - loss: 493.0938 - val_loss: 2247.9797
Epoch 5/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - loss: 448.2698 - val_loss: 709.1046
Epoch 6/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 417.5578 - val_loss: 1226.3580
Epoch 7/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - loss: 400.7864 - val_loss: 2680.9443
Epoch 8/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - loss: 385.4716 - val_loss: 5870.3071
Epoch 9/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - loss: 374.4704 - val_loss: 3624.4858
Epoch 10/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - loss: 361.9896 - val_loss: 5779.8149
Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 111.5575 

d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Fraction 0.2: RMSE=23.52, R2=0.68


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Fraction 0.3: RMSE=39.76, R2=0.10


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Fraction 0.5: RMSE=25.20, R2=0.64


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Fraction 0.7: RMSE=20.35, R2=0.76
